# Quickstart

A simple guide to computing EVI zonal statistics using Google Earth Engine or local computation.

In [1]:
import evy
import attaviz

attaviz.enable()

## Load Administrative Boundaries

`evy` aggregates EVI over polygons you supply. Use `get_boundaries` to fetch administrative units from [GeoBoundaries](https://www.geoboundaries.org/) (cached locally after the first call), or `load_boundaries` to read your own shapefile, GeoJSON, or GeoPackage.

In [2]:
gdf = evy.get_boundaries("SMR", admin_level=1)
gdf.head()

Found legacy boundaries cache at /Users/farhanreynaldo/Documents/world-bank/git-repo/evy/notebooks/.evy/boundaries. evy now caches to /Users/farhanreynaldo/Library/Caches/evy/boundaries; you can safely delete the legacy directory.


,shapeName,shapeISO,shapeID,shapeGroup,shapeType,geometry
0,Acquaviva,SM-01,276375B74977268893733,SMR,ADM1,"POLYGON ((12.43652 43.95663, 12.43577 43.95669..."
1,Borgo Maggiore,SM-06,276375B67153494303952,SMR,ADM1,"POLYGON ((12.43184 43.94467, 12.43239 43.94453..."
2,Chiesanuova,SM-02,276375B88489846739211,SMR,ADM1,"POLYGON ((12.41513 43.93117, 12.41537 43.93017..."
3,Domagnano,SM-03,276375B27851101452101,SMR,ADM1,"POLYGON ((12.47571 43.93733, 12.47807 43.93855..."
4,Faetano,SM-04,276375B67355364129568,SMR,ADM1,"POLYGON ((12.4724 43.92261, 12.47267 43.92243,..."


## Basic Usage

Pass the `GeoDataFrame` to `zonal_stats` along with `zone_col`, which is the column in your boundaries that identifies each zone. For GeoBoundaries data that's `shapeName`. The simplest call returns monthly EVI means for the past year:

In [3]:
df = evy.zonal_stats(gdf, zone_col="shapeName")
df.head()

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


,date,shapeName,mean
0,2025-09-01,Acquaviva,0.328136
1,2025-09-01,Borgo Maggiore,0.341599
2,2025-09-01,Chiesanuova,0.381155
3,2025-09-01,Domagnano,0.295232
4,2025-09-01,Faetano,0.345812


## Choose Data Source

The default data source is MODIS at 250m resolution. By default, zonal statistics are masked to cropland areas using Dynamic World land cover. Disable this with `mask_cropland=False`. 

Currently, we only support a single land cover dataset, but we're planning to add more in the future.

In [4]:
df_modis = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    source="modis",
    start_date="2024-01-01",
    end_date="2024-12-31",
    freq=evy.MONTHLY,  # or freq="ME"
    stats=["mean", "std", "min", "max"],
)

df_modis.head()

,date,shapeName,mean,std,min,max
0,2024-01-01,Acquaviva,0.292595,0.055530,0.19545,0.41040
1,2024-01-01,Borgo Maggiore,0.258617,0.048525,0.16480,0.38020
2,2024-01-01,Chiesanuova,0.250896,0.046290,0.17960,0.36435
3,2024-01-01,Domagnano,0.241463,0.039255,0.13940,0.33230
4,2024-01-01,Faetano,0.238605,0.036591,0.18535,0.34455


We also support Sentinel-2 for higher (10m) resolution:

In [5]:
df_s2 = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    source="sentinel2",
    start_date="2024-01-01",
    end_date="2024-12-31",
    freq=evy.MONTHLY,
)

df_s2.head()

,date,shapeName,mean
0,2024-01-01,Acquaviva,0.297950
1,2024-01-01,Borgo Maggiore,0.292187
2,2024-01-01,Chiesanuova,0.301793
3,2024-01-01,Domagnano,0.265209
4,2024-01-01,Faetano,0.268639


## Local Backend (No GEE Required)

Set `backend="local"` to compute EVI zonal statistics without Google Earth Engine. This uses Microsoft's Planetary Computer STAC catalog and it requires no authentication. Currently, the local backend only supports MODIS EVI with ESA WorldCover cropland masking, but we're planning to add Sentinel-2 and more land cover datasets in the future.

The local backend method consists of searching for MODIS data via STAC, loading rasters lazily with `odc-stac`, applying quality masking and EVI scaling, masking cropland using ESA WorldCover (class 40), and computing zonal stats with `exactextract` (partial-pixel weighting).

Both backends return the same columns: `date` (start of each period), your `zone_col`, and one column per statistic (`mean`, `std`, ...).

In [6]:
df_local = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    backend="local",
    start_date="2023-01-01",
    end_date="2023-12-31",
    freq=evy.MONTHLY,
    stats=["mean", "median"],
    include_geometry=True,
)

df_local.head()

,date,shapeName,mean,median,geometry
0,2023-01-01,Acquaviva,0.293736,0.299194,"POLYGON ((12.43652 43.95663, 12.43577 43.95669..."
1,2023-01-01,Borgo Maggiore,0.292080,0.281930,"POLYGON ((12.43184 43.94467, 12.43239 43.94453..."
2,2023-01-01,Chiesanuova,0.264241,0.270205,"POLYGON ((12.41513 43.93117, 12.41537 43.93017..."
3,2023-01-01,Domagnano,0.308663,0.311149,"POLYGON ((12.47571 43.93733, 12.47807 43.93855..."
4,2023-01-01,Faetano,0.302328,0.307069,"POLYGON ((12.4724 43.92261, 12.47267 43.92243,..."


## Low-Level Local Pipeline

For full control over the local computation pipeline, e.g. to inspect the raw EVI raster or the raw cropland mask, you can call the three main steps separately:

1. `load_modis`: STAC search + lazy `xarray.Dataset` with `evi_raw` and `qa` bands
2. `load_landcover`: aligned ESA WorldCover raster for cropland masking
3. `compute_zonal_stats`: full processing pipeline (quality mask → scale → temporal aggregation → optional cropland mask → extraction)

This is equivalent to `zonal_stats(backend="local")` but exposes the intermediate `xarray` objects.

In [7]:
ds_evi = evy.load_modis(gdf, start_date="2023-01-01", end_date="2023-12-31")
land_cover = evy.load_landcover(gdf, ds_evi)

df_pipeline = evy.compute_zonal_stats(
    ds_evi,
    gdf,
    zone_col="shapeName",
    land_cover=land_cover,
    freq=evy.MONTHLY,
    stats=["mean", "std"],
)
df_pipeline.head()

,date,shapeName,mean,std
0,2023-01-01,Acquaviva,0.293736,0.055950
1,2023-01-01,Borgo Maggiore,0.292080,0.053641
2,2023-01-01,Chiesanuova,0.264241,0.020815
3,2023-01-01,Domagnano,0.308663,0.040729
4,2023-01-01,Faetano,0.302328,0.034870


## Custom Boundaries

Use your own shapefile, GeoJSON, or GeoPackage with `evy.load_boundaries`. Pass the name of any column that identifies each zone as `zone_col`:

In [8]:
# gdf_custom = evy.load_boundaries("path/to/your/boundaries.shp")
# df_custom = evy.zonal_stats(
#     gdf_custom,
#     zone_col="admin_name",  # any column in your file
#     source="modis",
#     start_date="2024-01-01",
#     end_date="2024-12-31",
#     freq=evy.MONTHLY,
# )

## Large Jobs - Export to Google Drive

For queries spanning many years, export to Drive instead of downloading directly. Only supported with `backend="gee"`.

In [9]:
# This starts an async export task
# task_id = evy.zonal_stats(
#     gdf,
#     zone_col="shapeName",
#     start_date="2010-01-01",
#     end_date="2024-12-31",
#     freq=evy.YEARLY,
#     mask_cropland=True,
#     export_to_drive=True,
#     drive_folder="evy_exports",
# )
#
# print(f"Export started! Task ID: {task_id}")
# print("Check Google Drive for results when complete.")

# Check task status
# status = evy.check_task_status(task_id)
# print(status)

## Available Collections

See what data sources are available:

In [10]:
evy.list_collections()

,id,name,satellite,indices,resolution,temporal,start_date,end_date
0,MODIS/061/MOD13Q1,Terra MODIS Vegetation Indices,Terra,[evi],250,16-day,2000-02-18,present
1,MODIS/061/MYD13Q1,Aqua MODIS Vegetation Indices,Aqua,[evi],250,16-day,2002-07-04,present
2,COPERNICUS/S2_SR_HARMONIZED,Sentinel-2 MSI Surface Reflectance,Sentinel-2,[evi],10,5-day,2017-03-28,present


## Phenology Extraction (SOS, MOS, EOS)

We provide a phenology extraction method to extract crop seasonality metrics: Start of Season (SOS), Middle of Season (MOS), and End of Season (EOS).

The `value_col` argument must match the stat column in your input, for example `"mean"`. The name is the same for both backends.

In [11]:
phenology = evy.calculate_phenology(df_modis, value_col="mean")
phenology.head()

,month,value,smoothed,sos,mos,eos
0,1,0.252752,0.2332,2,4,12
1,2,0.302704,0.3431,2,4,12
2,3,0.427691,0.4235,2,4,12
3,4,0.509088,0.4935,2,4,12
4,5,0.477426,0.4919,2,4,12


Calculate the phenology metrics per governorate:

In [12]:
phenology_by_region = evy.calculate_phenology(
    df_modis, value_col="mean", group_col="shapeName"
)
phenology_by_region.head(12)

,month,value,smoothed,sos,mos,eos,shapeName
0,1,0.292595,0.2774,2,4,7,Acquaviva
1,2,0.371757,0.4072,2,4,7,Acquaviva
2,3,0.505443,0.4901,2,4,7,Acquaviva
3,4,0.541285,0.5463,2,4,7,Acquaviva
4,5,0.505662,0.4892,2,4,7,Acquaviva
5,6,0.366169,0.3805,2,4,7,Acquaviva
6,7,0.282310,0.2768,2,4,7,Acquaviva
7,8,0.246069,0.2598,2,4,7,Acquaviva
8,9,0.313517,0.3110,2,4,7,Acquaviva
9,10,0.380745,0.3704,2,4,7,Acquaviva


Filter data to the growing season only:

In [13]:
df_growing = evy.filter_growing_season(df_modis, start_month=2, end_month=6)
df_growing.head()

,date,shapeName,mean,std,min,max,year
0,2024-02-01,Acquaviva,0.371757,0.069425,0.22275,0.50190,2024
1,2024-02-01,Borgo Maggiore,0.307664,0.049014,0.20330,0.42460,2024
2,2024-02-01,Chiesanuova,0.300579,0.049501,0.21860,0.39475,2024
3,2024-02-01,Domagnano,0.284428,0.051244,0.13045,0.41590,2024
4,2024-02-01,Faetano,0.275472,0.041698,0.20265,0.42120,2024


## Visualizations

Built-in interactive charts using Altair:

In [14]:
chart = evy.plot_seasonality(phenology)
chart

alt.LayerChart(...)

In [15]:
chart_regions = evy.plot_seasonality_by_region(
    phenology_by_region, region_col="shapeName"
)
chart_regions

alt.FacetChart(...)